In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 1. knowlege base 구성을 위한 데이터 생성

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # character 단위로 짜름
loader = Docx2txtLoader('tax_docx/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, #문서를 쪼갤 떄 1500글자씩 chunking
        chunk_overlap=20, #200문자씩 overlap 하여 chunking 작업
        #separators=["\n\n","\n"," "] -> 기본값   : 다음의 순서대로 chunking 시도
)
documents = loader.load_and_split(text_splitter=text_splitter)
print('chunk 갯수: ', len(documents))

chunk 갯수:  180


In [5]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_markdown.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 

In [8]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env 덮어쓰기
embedding = UpstageEmbeddings(
                    model ='embedding-query'
)


In [9]:
len(embedding.embed_query("소득세법"))

4096

In [13]:
from pinecone import Pinecone, ServerlessSpec
pc=Pinecone()
[index.name for index in pc.list_indexes()]

['tax-index', 'tax-index-table', 'tax-index-openai']

In [15]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec
# pinecone 객체 생성
pc=Pinecone()  # api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-markdown'
dimension = len(embedding.embed_query("차원확인용"))

# 2. 인덱스 생성(해당인덱스가 없는경우)
existing_indexes = [index.name for index in pc.list_indexes()]
#print(index_name in existing_indexes)
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine', #유사도 측정방식
        spec=ServerlessSpec(
            cloud='aws',    #무료 플랜은 aws
            region='us-east-1' # 무료 플랜은 us-east-1만 가능
        )
    )
    #인덱스가 준비 될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) #1초 대기
    print(f'인덱스{index_name}를 생성해였습니다')    
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

인덱스 tax-index-markdown이 이미 존재합니다


In [16]:
[index.name for index in pc.list_indexes()]

['tax-index', 'tax-index-markdown', 'tax-index-table', 'tax-index-openai']

In [17]:
index = pc.Index(index_name)
print(index.describe_index_stats())

C:\Users\mbc\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [19]:
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# 처음 한번
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
pc=Pinecone()
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할때 
index_name='tax-index-table'
database =PineconeVectorStore(
    embedding=embedding,
    index_name = index_name
)

CPU times: total: 9.77 s
Wall time: 26.9 s


# 2. 답변 생성 전 retrieval 확인


In [21]:
query= "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,k=3) # 기본 k값은 4

In [22]:
retrieved_docs[0]

Document(id='397fc970-d704-487f-b122-b1713c68bb18', metadata={'source': 'tax_docx/소득세법_with_markdown.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1

In [23]:
retriever = database.as_retriever(search_type='similarity',
                                 search_kwarg={'k':2}
                                 )
retriever_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")
retriever_docs[0]

Document(id='397fc970-d704-487f-b122-b1713c68bb18', metadata={'source': 'tax_docx/소득세법_with_markdown.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1

In [26]:
retrieved_doc="\n--\n".join([doc.page_content for doc in retriever_docs])
'55조' in retrieved_doc
#print(retriever_doc)''

True

# 3. 답변 생성


In [28]:
from openai import OpenAI
client= OpenAI()
query= "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role':'system','content':'당신은 최고의 한국 소득세법 전문가입니다'},
            {
                'role':'user',
                'content':f"""-[context]를 참고해서 사용자의 질문에 10줄이내로 답변해 주세요.
                -[context]:{retrieved_doc}
                -질문 : {query}"""
            }
        ],
    temperature=0.2
)

In [29]:
result=response.choices[0].message

In [30]:
print(result)

ChatCompletionMessage(content='연봉이 5천만원인 직장인의 소득세를 계산하기 위해서는 종합소득 과세표준과 세율을 적용해야 합니다.\n\n1. **과세표준**: 5천만원\n2. **세율**: 5천만원 초과 8,800만원 이하의 구간에 해당하므로, 세율은 24%입니다.\n3. **세액 계산**:\n   - 5천만원 초과 부분: 5천만원 초과 금액 = 5천만원 - 5천만원 = 0\n   - 기본 세액: 624만원 (5천만원 초과 구간의 기본 세액)\n   - 추가 세액: 0 (5천만원 초과 금액이 없으므로)\n\n따라서, 총 소득세는 624만원입니다. 추가 공제나 세액공제를 고려하지 않은 단순 계산입니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [31]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
        model='gpt-4.1-mini'
)

In [33]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
        model='solar-pro4',
        reasoning_effort='low' #느리지만 더 깊게 추론하여 답변(low,medium,high)
)

In [34]:
prompt=f"""[identity]
-당신은 최고의 한국 소득세법 전문가입니다.
-[context]를 참고하여 사용자의 질문에 답변해주세요. [context]는 다음과 같아요
{retriever_doc}
- 질문 : {query}"""

In [35]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

질문에 답변드리기 위해, 제공된 [context]의 법조문을 바탕으로 **2024~2025년 현행 소득세법 기준**으로 계산 절차를 설명드리겠습니다. 다만, 실제 납부세액은 부양가족 유무, 신용카드·의료비·보험료 등 특별공제, 연금계좌 납입액, 기타 세액공제 적용 여부에 따라 달라지므로 아래 계산은 **다른 공제·감면이 없는 최소한의 가정** 하에 산정한 참고치임을 먼저 밝힙니다.

---
### 🔹 계산 전제
- 연봉(총급여액): **5,000만원**
- 근로소득 외 다른 종합소득 없음
- 기본공제 대상 부양가족 없음(본인에 대한 기본공제만 적용)
- 연금계좌 납입액, 보험료, 의료비, 교육비, 기부금, 자녀 등 기타 공제·세액공제 미적용 가정

---
### 1️⃣ 근로소득공제 적용 (소득세법 제47조)
총급여액에서 근로소득공제를 차감하여 `근로소득금액`을 산출합니다. 현행 근로소득공제율(4,500만원 초과 1억원 이하 구간)은 다음과 같습니다.
> **1,200만원 + (총급여액 - 4,500만원) × 5%**

- 5,000만원 기준:  
  `1,200만원 + (5,000만원 - 4,500만원) × 5%`  
  = 1,200만원 + 25만원 = **1,225만원**
- 근로소득금액 = 5,000만원 - 1,225만원 = **3,775만원**

---
### 2️⃣ 종합소득공제 적용 (문맥 제50조 기본공제 등)
`종합소득과세표준` = 근로소득금액 - 종합소득공제  
가장 기본적인 본인 기본공제(1인당 연 150만원, 제50조제1항제1호)만 적용하면:
- 과세표준 = 3,775만원 - 150만원 = **3,625만원**

---
### 3️⃣ 산출세액 계산 (문맥 제55조 세율표 적용)
과세표준 3,625만원은 `1,400만원 초과 5,000만원 이하` 구간에 해당합니다.
> `84만원 + (1,400만원을 초과하는 금액의 15%)`

- 초과금액: 3,625만원 - 1,400만원 = 2,225만원
- 산출세액 = 84만원 + (2,225만원 × 15

In [36]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = UpstageEmbeddings(model = "embedding-query")
llm = ChatUpstage(model = 'solar-pro4')
#2. vector DB load
#pc = Pinecone()
index_name= 'tax-index-table'
vectorStore=PineconeVectorStore(
            embedding=embedding,
            index_name=index_name
                )
#3. Retriever 생성 
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':2} # 유사도가 높은 top-2추출
)

#4. 프롬프트템플릿
template= f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질몬에 답하세요.
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return '\n--\n'.join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])

In [37]:
# 6. RAG 체인 구성(LCEL 방식)
rag_chain = (
    {
        "context":retriever|format_documents,
        "query" : RunnablePassthrough() # 질문만 그대로 전달
    }
    |prompt
    |llm
    |StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'연봉 5천만원만으로는 정확한 소득세를 계산할 수 없습니다. 소득공제 및 세액공제 적용 여부에 따라 과세표준과 세액이 달라지므로, 제시된 세율표상 산출세액만으로는 확정되는 세액을 알 수 없기 때문입니다. 정확한 금액은 근로소득공제, 인적공제, 연금보험료 공제 등 구체적 사항이 반영되어야 산출 가능합니다.'

# 4. 키워드 사전 활용

In [38]:
query='연봉 5천만원인 직장인의 소득세는 얼마인가요?'
query='종합소득이 5,000만원인 거주자의 소득세는 얼마인가요?'

In [40]:
import numpy as np
def cosine_similarity(vec1,vec2):
    '두 벡터의 코사인 유사도를 반환'
    dot_product= np.dot(vec1,vec2)
    norm_vec1=np.linalg.norm(vec1)
    norm_vec2=np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0
    return dot_product / (norm_vec1*norm_vec2)
vec1= embedding.embed_query("직장인")
vec2= embedding.embed_query("거주자")
cosine_similarity(vec1,vec2)

np.float64(0.5402861885922909)

In [43]:
vec1= embedding.embed_query("5000")
vec2= embedding.embed_query("5,000")
cosine_similarity(vec1,vec2)

np.float64(0.9712007742214392)

In [46]:
# 키워드 사전 : 일반사용자 단어 -> 소득세법 용어
dictionary = ["연봉-> 종합소득", "사람을 나타내는 표현 -> 거주자","5천->5,000"]
prompt =ChatPromptTemplate.from_template(
                                    f'''사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을 변경해 주세요.
                                    만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만
                                    리턴해 주세요
                                    -사전 : {dictionary}
                                    -질문 : {{question}}''')


In [47]:
diction_chain = prompt

In [58]:
strOutputParser = StrOutputParser()   
strOutputParser.invoke(llm.invoke(prompt.invoke({"question":"소득이 높은 남자가 있습니다"})))

'종합소득이 높은 거주자가 있습니다'

In [62]:
# 위를 langchain으로
dictionary_chain = prompt | llm | StrOutputParser()
dictionary_chain.invoke({"question":"연봉이 5천만원인 직장인 소득세는 얼마예요???"})

'종합소득세가 5,000만원인 거주자 소득세는 얼마예요?'

In [60]:
new_chain = dictionary_chain | rag_chain

In [63]:
new_chain.invoke("연봉이 5천만원인 직장인의 소득세는 얼마예요?")

"주어진 문맥의 제55조 제1항 세율표에 따르면, 종합소득 과세표준이 정확히 5,000만원일 경우 산출세액은 624만원 + (5,000만원 - 1,400만원) × 15% = 1,179만원입니다. 다만, 질문의 '종합소득 5,000만원'이 소득공제 전의 총수입금액인지 과세표준인지 명확하지 않으며, 실제 납부세액은 다양한 소득공제와 세액공제를 적용한 후 결정되므로 정확한 금액을 특정할 수 없습니다. 따라서 단순히 과세표준이 5,000만원이라고 가정할 경우의 산출세액은 1,179만원으로 계산됩니다."

In [ ]:
# 입력{"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요??"}
#              ↓
# dictionary_chain에 의해 질문 수정(document용어 수정)
#              ↓
#        {"개선된 질문"}
#              ↓
#          retriever             어떤분류의 질문이기에 분류해서 ~줄였다
#              ↓
#       format_documents
#              ↓
#       rag_chain의 prompt  +llm
#              ↓
#       strOutputParser()


